# Enhanced Metrics for Amazon SageMaker AI Endpoints

This notebook demonstrates how to use enhanced metrics to monitor, optimize, and troubleshoot SageMaker AI endpoints.

**What you'll learn:**
- Deploy an endpoint with enhanced metrics enabled
- Track model copies over time during auto-scaling
- Calculate cost per model and cost per 1K invocations
- View routing balance across instances
- Identify under-utilized model copies

**Prerequisites:**
- AWS account with SageMaker permissions
- IAM role with CloudWatch read access
- Sufficient quota for ml.g5.12xlarge instances

## Setup and Configuration

In [ ]:
%pip install boto3 --upgrade --quiet
%pip install sagemaker==2.257.0 #use v2 sdk

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import boto3
import sagemaker
import json
import time
import random
import string
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# Initialize clients
sess = sagemaker.session.Session()
region = sess.boto_region_name
role = sagemaker.get_execution_role()

sm_client = boto3.client('sagemaker')
smr_client = boto3.client('sagemaker-runtime')
cloudwatch = boto3.client('cloudwatch')

print(f"Region: {region}")
print(f"Role: {role}")

## Configuration Variables

In [ ]:
# Endpoint configuration
SALT = ''.join(random.choices(string.ascii_letters + string.digits, k=6))
ENDPOINT_BASE_NAME = 'enhanced-metrics-demo'
ENDPOINT_NAME = f"{ENDPOINT_BASE_NAME}-{SALT}"
ENDPOINT_CONFIG_NAME = f'{ENDPOINT_NAME}-config'
# model_name = f'{ENDPOINT_BASE_NAME}-model'
# IC_NAME_A = f'{ENDPOINT_NAME}-A-ic'
# IC_NAME_B = f'{ENDPOINT_NAME}-B-ic'

# # Model configuration
# MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
# INSTANCE_TYPE = 'ml.g6.12xlarge'
# INITIAL_INSTANCE_COUNT = 2

# GPU_COUNT = 4

# Model configuration
MODEL_A_ID = 'openai/gpt-oss-20b'
MODEL_A_NAME = f'{ENDPOINT_NAME}-gpt-oss'
IC_NAME_A = f'IC-{MODEL_A_NAME}'

MODEL_B_ID = 'Qwen/Qwen2.5-7B-Instruct'
MODEL_B_NAME = f'{ENDPOINT_NAME}-Qwen'
IC_NAME_B = f'IC-{MODEL_B_NAME}'


INSTANCE_TYPE = 'ml.g6.12xlarge'
INITIAL_INSTANCE_COUNT = 2

MODEL_A_GPU_COUNT = 1
MODEL_B_GPU_COUNT = 4

# Enhanced metrics settings
ENABLE_ENHANCED_METRICS = True
METRICS_PUBLISH_FREQUENCY = 10  # seconds (10, 30, or 60)

# Instance pricing (on-demand, us-east-1)
INSTANCE_PRICES = {
    'ml.g6.12xlarge': 5.752
}
INSTANCE_HOURLY_COST = INSTANCE_PRICES.get(INSTANCE_TYPE, 5.752)

## Part 1: Deploy Endpoint with Inference Components and Enhanced Metrics

In [ ]:
# Get the HuggingFace TGI container image
from sagemaker import image_uris

# Retrieve the LMI image URI
image_uri = image_uris.retrieve(
    framework='djl-lmi',
    region=region,
    version='0.36.0',
    instance_type=INSTANCE_TYPE
)
print(f"Image URI: {image_uri}")

### Model A - openai/gpt-oss-20b 

In [ ]:
# LMI environment configuration
lmi_config = {
    'HF_MODEL_ID': MODEL_A_ID,
    'OPTION_MAX_MODEL_LEN': '8192',
    'OPTION_TENSOR_PARALLEL_DEGREE': '4',
    'HF_TOKEN': '<HF_TOKEN_HERE>'
}

try:
    sm_client.delete_model(ModelName=MODEL_A_NAME)
except: pass

sm_client.create_model(
    ModelName=MODEL_A_NAME,
    ExecutionRoleArn=role,
    PrimaryContainer={'Image': image_uri, 'Environment': lmi_config}
)
print(f"Created model: {MODEL_A_NAME}")

### Model B - Qwen/Qwen2.5-7B-Instruct

In [ ]:
# LMI environment configuration
lmi_config = {
    'HF_MODEL_ID': MODEL_B_ID,
    'OPTION_MAX_MODEL_LEN': '8192',
    'OPTION_TENSOR_PARALLEL_DEGREE': '1',
    'HF_TOKEN': '<HF_TOKEN_HERE>'
}

try:
    sm_client.delete_model(ModelName=MODEL_B_NAME)
except: pass

sm_client.create_model(
    ModelName=MODEL_B_NAME,
    ExecutionRoleArn=role,
    PrimaryContainer={'Image': image_uri, 'Environment': lmi_config}
)
print(f"Created model: {MODEL_B_NAME}")

### Create Endpoint

In [ ]:
# Create endpoint config for inference components
try:
    sm_client.delete_endpoint_config(EndpointConfigName=ENDPOINT_CONFIG_NAME)
except: pass

sm_client.create_endpoint_config(
    EndpointConfigName=ENDPOINT_CONFIG_NAME,
    ExecutionRoleArn=role,
    ProductionVariants=[{
        'VariantName': 'AllTraffic',
        # 'ModelName': model_name,
        'InstanceType': INSTANCE_TYPE,
        'InitialInstanceCount': INITIAL_INSTANCE_COUNT,
        'RoutingConfig': {'RoutingStrategy': 'LEAST_OUTSTANDING_REQUESTS'}
    }],
    # KEY: Enable enhanced metrics for container-level visibility
    MetricsConfig={
        'EnableEnhancedMetrics': ENABLE_ENHANCED_METRICS,
        'MetricPublishFrequencyInSeconds': METRICS_PUBLISH_FREQUENCY
    }
)
print(f"Created {ENDPOINT_CONFIG_NAME} config with enhanced metrics (frequency: {METRICS_PUBLISH_FREQUENCY}s)")

In [ ]:
# Create and wait for endpoint
try:
    sm_client.delete_inference_component(InferenceComponentName=IC_NAME_A)
    sm_client.get_waiter('inference_component_deleted').wait(InferenceComponentName=IC_NAME_A)
except: pass
try:
    sm_client.delete_inference_component(InferenceComponentName=IC_NAME_B)
    sm_client.get_waiter('inference_component_deleted').wait(InferenceComponentName=IC_NAME_B)
except: pass
try:
    sm_client.delete_endpoint(EndpointName=ENDPOINT_NAME)
    sm_client.get_waiter('endpoint_deleted').wait(EndpointName=ENDPOINT_NAME)
except: pass

sm_client.create_endpoint(EndpointName=ENDPOINT_NAME, EndpointConfigName=ENDPOINT_CONFIG_NAME)
print(f"Creating endpoint {ENDPOINT_NAME}... (5-10 min)")

sm_client.get_waiter('endpoint_in_service').wait(
    EndpointName=ENDPOINT_NAME,
    WaiterConfig={'Delay': 30, 'MaxAttempts': 60}
)
print("Endpoint is InService!")

### Create Inference Component (1 GPU per copy) - Model A openai/gpt-oss-20b

In [ ]:
# Create inference component with 1 GPU per copy (4 copies per instance)
try:
    sm_client.delete_inference_component(InferenceComponentName=IC_NAME_A)
    sm_client.get_waiter('inference_component_deleted').wait(InferenceComponentName=IC_NAME_A)
except: pass

sm_client.create_inference_component(
    InferenceComponentName=IC_NAME_A,
    EndpointName=ENDPOINT_NAME,
    VariantName='AllTraffic',
    Specification={
        'ModelName': MODEL_A_NAME,
        'ComputeResourceRequirements': {
            'NumberOfAcceleratorDevicesRequired': 4,
            'MinMemoryRequiredInMb': 8192 // 2
        }
    },
    RuntimeConfig={
        'CopyCount': 1
    }
)

print(f"Creating inference component with 4 copies (1 GPU each)...")

In [ ]:
# Wait for inference component to be ready
while True:
    response = sm_client.describe_inference_component(InferenceComponentName=IC_NAME_A)
    status = response['InferenceComponentStatus']
    print(f"Status: {status}")
    if status == 'InService':
        break
    elif status == 'Failed':
        print(f"Failed: {response.get('FailureReason', 'Unknown')}")
        break
    time.sleep(30)

print(f"\n✅ Inference component ready!")
print(f"Configuration: 4 model copies, 1 GPU per copy")

### Create Inference Component (1 copy per endpoint) - Model B Qwen/Qwen2.5-7B-Instruct

In [ ]:
# Create inference component with 4 GPUs per copy (1 copy per instance)
try:
    sm_client.delete_inference_component(InferenceComponentName=IC_NAME_B)
    sm_client.get_waiter('inference_component_deleted').wait(InferenceComponentName=IC_NAME_B)
except: pass

sm_client.create_inference_component(
    InferenceComponentName=IC_NAME_B,
    EndpointName=ENDPOINT_NAME,
    VariantName='AllTraffic',
    Specification={
        'ModelName': MODEL_B_NAME,
        'ComputeResourceRequirements': {
            'NumberOfAcceleratorDevicesRequired': 1,
            'MinMemoryRequiredInMb': 8192 * 4 // 2
        }
    },
    RuntimeConfig={
        'CopyCount': 2
    }
)

print(f"Creating inference component with 2 copies (1 GPU each)...")

In [ ]:
# Wait for inference component to be ready
while True:
    response = sm_client.describe_inference_component(InferenceComponentName=IC_NAME_B)
    status = response['InferenceComponentStatus']
    print(f"Status: {status}")
    if status == 'InService':
        break
    elif status == 'Failed':
        print(f"Failed: {response.get('FailureReason', 'Unknown')}")
        break
    time.sleep(30)

print(f"\n✅ Inference component ready!")
print(f"Configuration: 2 model copies, 1 GPU per copy")

## Part 2: Generate Traffic

In [ ]:
from endpoint_metrics_helper import invoke_endpoint

prompts = [
    "Explain machine learning in simple terms.",
    "What is the difference between AI and ML?",
    "How do neural networks work?",
    "What are transformers in deep learning?"
]

# Test single request
print(invoke_endpoint(prompts[0], ENDPOINT_NAME, IC_NAME_A)['generated_text'][:200])
print(invoke_endpoint(prompts[3], ENDPOINT_NAME, IC_NAME_B)['generated_text'][:200])

In [ ]:
# Generate concurrent load
from endpoint_metrics_helper import generate_load

print("\n" + "="*60)
print("LOAD GENERATION")
print("="*60)
print(f"Sending to both ICs simultaneously...")
print(f"  • IC A: {IC_NAME_A}")
print(f"  • IC B: {IC_NAME_B}")
print(f"")

# Use delay_between_requests to control traffic pattern:
# - 0.0 = burst mode (all requests submitted immediately)
# - 0.1 = 10 request pairs/second submission rate
# - 0.2 = 5 request pairs/second (more even distribution)
# - 0.5 = 2 request pairs/second (very even distribution)
result = generate_load(ENDPOINT_NAME, [IC_NAME_A, IC_NAME_B], 1000, 25, 0.2)

print(f"\n" + "="*60)
print(f"LOAD GENERATION COMPLETE")
print(f"="*60)
print(f"✓ Successful: {result['success']:,} requests")
print(f"✗ Errors: {result['error']:,} requests")
print(f"Total: {result['success'] + result['error']:,} requests")
print(f"Success Rate: {(result['success']/(result['success']+result['error'])*100):.1f}%")
print(f"="*60)

## Part 3: Create Metrics Dashboards

In [ ]:
from endpoint_metrics_helper import create_metrics_widget

create_metrics_widget('enhanced-metrics-demo')

In [ ]:
from endpoint_metrics_helper import create_metrics_widget, create_cost_widget, get_model_cost

# For cost metrics with widget
create_cost_widget(IC_NAME_A, cost_per_hour=INSTANCE_HOURLY_COST)
create_cost_widget(IC_NAME_B, cost_per_hour=INSTANCE_HOURLY_COST)

In [ ]:
# Or get cost programmatically
result = get_model_cost(IC_NAME_A, INSTANCE_HOURLY_COST, time_range='Last 1 hour')
print(f"Cumulative cost for {IC_NAME_A}: ${result['cumulative_cost']:.4f}")

result = get_model_cost(IC_NAME_B, INSTANCE_HOURLY_COST, time_range='Last 1 hour')
print(f"Cumulative cost for {IC_NAME_B}: ${result['cumulative_cost']:.4f}")

## Part 4: Query Enhanced Metrics

In [ ]:
from endpoint_metrics_helper import get_current_model_copies

# Get current configuration
copy_info_a = get_current_model_copies(IC_NAME_A)
if copy_info_a:
    print(f"{IC_NAME_A} Configuration:")
    print(f"  Status: {copy_info_a['status']}")
    print(f"  Current Model Copies: {copy_info_a['current_copy_count']}")
    print(f"  Desired Model Copies: {copy_info_a['desired_copy_count']}")
    print(f"  GPUs per Copy: {copy_info_a['gpus_per_copy']}")
    print(f"  Total GPUs in use: {copy_info_a['current_copy_count'] * copy_info_a['gpus_per_copy']}")

copy_info_b = get_current_model_copies(IC_NAME_B)
if copy_info_b:
    print(f"{IC_NAME_B} Configuration:")
    print(f"  Status: {copy_info_b['status']}")
    print(f"  Current Model Copies: {copy_info_b['current_copy_count']}")
    print(f"  Desired Model Copies: {copy_info_b['desired_copy_count']}")
    print(f"  GPUs per Copy: {copy_info_b['gpus_per_copy']}")
    print(f"  Total GPUs in use: {copy_info_b['current_copy_count'] * copy_info_b['gpus_per_copy']}")

### 4.1 Cost Per 1K Invocations w/Model Copies

In [ ]:
from endpoint_metrics_helper import calculate_cost_per_1k

cost_copies_A = calculate_cost_per_1k(IC_NAME_A, ENDPOINT_NAME, MODEL_A_GPU_COUNT, INSTANCE_HOURLY_COST)
cost_copies_B = calculate_cost_per_1k(IC_NAME_B, ENDPOINT_NAME, MODEL_B_GPU_COUNT, INSTANCE_HOURLY_COST)

cost_data = {IC_NAME_A: cost_copies_A, IC_NAME_B: cost_copies_B}

for name, cost_copies in cost_data.items():
    print(f"*********{name}*********")
    print(f"Overall Metrics:")
    print(f"  Total invocations: {cost_copies['invocations']:,.0f}")
    print(f"  Total cost ({cost_copies['hours']}h): ${cost_copies['total_cost']:.2f}")
    print(f"  Cost per 1K invocations: ${cost_copies['cost_per_1k']:.4f}")
    
    if 'model_copies' in cost_copies:
        total_gpus = cost_copies['total_gpus_used']
        gpu_pct = cost_copies['gpu_fraction'] * 100
        print(f"\nResource Usage:")
        print(f"  Total GPUs used: {total_gpus} ({gpu_pct:.0f}% of instance)")
        print(f"  Configuration: {cost_copies['model_copies']} copies × {cost_copies['gpus_per_copy']} GPU(s) each")
        
        print(f"\nPer-Copy Breakdown:")
        print(f"  Invocations per copy: {cost_copies['invocations_per_copy']:,.0f}")
        print(f"  Cost per copy: ${cost_copies['cost_per_copy']:.2f}")
        print(f"  Cost per 1K per copy: ${cost_copies['cost_per_1k_per_copy']:.4f}")
        
        print(f"\n💡 Insight: Each of the {cost_copies['model_copies']} model copies handled ~{cost_copies['invocations_per_copy']:,.0f} requests")
        print(f"   Using {total_gpus} total GPUs ({gpu_pct:.0f}% of instance), you're paying ${cost_copies['cost_per_copy']:.2f}/copy for this period")
    print()
    print("ASSUMES NO SCALING EVENTS DURING INFERENCE TIME")

### 4.2 Routing Balance w/Model Copies

In [ ]:
from endpoint_metrics_helper import analyze_routing_detailed

# Analyze both ICs
routing_a = analyze_routing_detailed(IC_NAME_A, hours=1)
routing_b = analyze_routing_detailed(IC_NAME_B, hours=1)

# Display results showing hot copies
for ic_name, routing in [(IC_NAME_A, routing_a), (IC_NAME_B, routing_b)]:
    if not routing:
        continue
    
    print(f"\n{'='*80}")
    print(f"Traffic Routing Analysis: {ic_name}")
    print(f"{'='*80}")
    
    if 'model_copies' in routing:
        print(f"\nConfiguration: {routing['model_copies']} copies × {routing['gpus_per_copy']} GPU(s)")
    
    if 'total_invocations' in routing:
        total = routing['total_invocations']
        print(f"\nTotal Invocations: {total['sum']:,.0f}")
    
    if 'container_invocations' in routing:
        containers = routing['container_invocations']
        print(f"\nPer-Copy Traffic (sorted by hottest):")
        print(f"{'Rank':<6} {'Instance':<12} {'Container':<14} {'GPU':<8} {'Total':<12} {'% of Total':<12}")
        print("-" * 70)
        
        total_sum = sum(c['total'] for c in containers)
        
        for idx, container in enumerate(containers, 1):
            pct = (container['total'] / total_sum * 100) if total_sum > 0 else 0
                        
            print(f"{idx:<6} {container['instance_id']:<12} {container['container_id']:<14} "
                  f"{container['gpu_id']:<8} {container['total']:<12,.0f} {pct:<11.1f}%")
        
    else:
        print("\n⚠️  No per-copy metrics available. Invocations may not have container-level dimensions.")
    
    print()

### 4.3 Resource Utilization

In [ ]:
from endpoint_metrics_helper import analyze_utilization_detailed

# Display results with full granularity
util_data = {}

for ic_name in [IC_NAME_A, IC_NAME_B]:
    util_data[ic_name] = analyze_utilization_detailed(ic_name, hours=1)

for ic_name, data in util_data.items():
    print(f"\n{'='*100}")
    print(f"{ic_name}")
    print(f"{'='*100}")
    
    gpu_metrics = data['gpu_metrics']
    
    # Group by instance
    instances = {}
    for metric in gpu_metrics:
        instance_id = metric['instance_id']
        if instance_id not in instances:
            instances[instance_id] = []
        instances[instance_id].append(metric)
    
    print(f"\nTotal: {len(instances)} instances, {len(gpu_metrics)} containers")
    
    # Display per instance
    for instance_id, metrics_list in sorted(instances.items()):
        print(f"\n📦 Instance: {instance_id} ({len(metrics_list)} containers)")
        
        # Sort by container and GPU for readability
        sorted_metrics = sorted(metrics_list, key=lambda x: (x['container_id'], x['gpu_id']))
        
        for metric in sorted_metrics:
            container_id = metric['container_id']
            gpu_id = metric['gpu_id']
            metrics = metric['metrics']
            
            print(f"  └─ Container {container_id} → {gpu_id}:")
            
            if 'GPUUtilizationNormalized' in metrics:
                vals = metrics['GPUUtilizationNormalized']
                print(f"     GPU Compute: Avg={vals['avg']:.1f}%, Max={vals['max']:.1f}%, Min={vals['min']:.1f}%")
            
            if 'GPUMemoryUtilizationNormalized' in metrics:
                vals = metrics['GPUMemoryUtilizationNormalized']
                print(f"     GPU Memory:  Avg={vals['avg']:.1f}%, Max={vals['max']:.1f}%, Min={vals['min']:.1f}%")
                
                # Recommendations
                if vals['avg'] > 85:
                    print(f"     ⚠️  High GPU memory - risk of OOM")
                elif vals['avg'] < 30:
                    print(f"     💡 Low GPU memory - underutilized")
    
    # Summary by GPU across all instances
    print(f"\n{'─'*100}")
    print(f"GPU Summary (aggregated across all instances/containers)")
    print(f"{'─'*100}")
    
    gpu_summary = {}
    for metric in gpu_metrics:
        gpu_id = metric['gpu_id']
        if gpu_id not in gpu_summary:
            gpu_summary[gpu_id] = {'compute': [], 'memory': []}
        
        if 'GPUUtilizationNormalized' in metric['metrics']:
            gpu_summary[gpu_id]['compute'].append(metric['metrics']['GPUUtilizationNormalized']['avg'])
        if 'GPUMemoryUtilizationNormalized' in metric['metrics']:
            gpu_summary[gpu_id]['memory'].append(metric['metrics']['GPUMemoryUtilizationNormalized']['avg'])
    
    for gpu_id in sorted(gpu_summary.keys()):
        compute_vals = gpu_summary[gpu_id]['compute']
        memory_vals = gpu_summary[gpu_id]['memory']
        
        print(f"\n{gpu_id} (across {len(compute_vals)} containers):")
        if compute_vals:
            print(f"  GPU Compute: Avg={sum(compute_vals)/len(compute_vals):.1f}%, "
                  f"Max={max(compute_vals):.1f}%, Min={min(compute_vals):.1f}%")
        if memory_vals:
            print(f"  GPU Memory:  Avg={sum(memory_vals)/len(memory_vals):.1f}%, "
                  f"Max={max(memory_vals):.1f}%, Min={min(memory_vals):.1f}%")
    
    # Display component-level metrics once
    print(f"\n{'─'*100}")
    print(f"Component-Level Metrics (shared across all instances)")
    print(f"{'─'*100}")
    
    comp_metrics = data['component_metrics']
    
    print("\nResource Utilization:")
    for metric in ['CPUUtilizationNormalized', 'MemoryUtilizationNormalized']:
        if metric in comp_metrics:
            vals = comp_metrics[metric]
            display_name = metric.replace('Normalized', '').replace('Utilization', ' Util')
            print(f"  {display_name}: Avg={vals['avg']:.1f}%, Max={vals['max']:.1f}%, Min={vals['min']:.1f}%")
    
    print("\nError Rates:")
    total_errors = 0
    for metric in ['Invocation4XXErrors', 'Invocation5XXErrors', 'InvocationModelErrors']:
        if metric in comp_metrics:
            count = comp_metrics[metric]['total']
            total_errors += count
            print(f"  {metric}: {count:.0f}")
    if total_errors == 0:
        print("  ✅ No errors detected")
    
    print("\nLatency Metrics:")
    for metric in ['ModelLatency', 'OverheadLatency']:
        if metric in comp_metrics:
            vals = comp_metrics[metric]
            print(f"  {metric}: Avg={vals['avg']:.0f}ms, P50={vals['p50']:.0f}ms, Max={vals['max']:.0f}ms")
    
    if 'ModelLatency' in comp_metrics and 'OverheadLatency' in comp_metrics:
        overhead_pct = (comp_metrics['OverheadLatency']['avg'] / comp_metrics['ModelLatency']['avg']) * 100
        print(f"\n  💡 Overhead is {overhead_pct:.1f}% of model latency")


### 4.4 Visualizations 

In [ ]:
import numpy as np
from enhanced_metrics_visualization import visualize_gpu_metrics

# Run visualization
visualize_gpu_metrics(util_data)

# Print summary statistics
print("\n" + "="*100)
print("GPU METRICS SUMMARY")
print("="*100)

for ic_name, data in util_data.items():
    print(f"\n{ic_name}:")
    gpu_metrics = data['gpu_metrics']
    
    print(f"  Total containers: {len(gpu_metrics)}")
    
    # Count instances
    instances = set(m['instance_id'] for m in gpu_metrics)
    print(f"  Total instances: {len(instances)}")
    
    # GPU summary
    gpu_summary = {}
    for metric in gpu_metrics:
        gpu_id = metric['gpu_id']
        if gpu_id not in gpu_summary:
            gpu_summary[gpu_id] = {'compute': [], 'memory': [], 'containers': 0}
        
        gpu_summary[gpu_id]['containers'] += 1
        
        if 'GPUUtilizationNormalized' in metric['metrics']:
            gpu_summary[gpu_id]['compute'].append(metric['metrics']['GPUUtilizationNormalized']['avg'])
        if 'GPUMemoryUtilizationNormalized' in metric['metrics']:
            gpu_summary[gpu_id]['memory'].append(metric['metrics']['GPUMemoryUtilizationNormalized']['avg'])
    
    print(f"\n  Per-GPU Summary:")
    for gpu_id in sorted(gpu_summary.keys()):
        compute_vals = gpu_summary[gpu_id]['compute']
        memory_vals = gpu_summary[gpu_id]['memory']
        containers = gpu_summary[gpu_id]['containers']
        
        print(f"    {gpu_id} ({containers} containers):")
        if compute_vals:
            print(f"      Compute: Avg={np.mean(compute_vals):.1f}%, "
                  f"Std={np.std(compute_vals):.1f}%, Range=[{min(compute_vals):.1f}-{max(compute_vals):.1f}]%")
        if memory_vals:
            print(f"      Memory:  Avg={np.mean(memory_vals):.1f}%, "
                  f"Std={np.std(memory_vals):.1f}%, Range=[{min(memory_vals):.1f}-{max(memory_vals):.1f}]%")


In [ ]:
from enhanced_metrics_visualization import visualize_gpu_utilization

visualize_gpu_utilization(util_data, [IC_NAME_A, IC_NAME_B], [MODEL_A_GPU_COUNT, MODEL_B_GPU_COUNT])

# Print summary
print("\n" + "="*100)
print("CONFIGURATION COMPARISON SUMMARY")
print("="*100)

for gpu_id in all_gpu_ids:
    print(f"\n{gpu_id}:")
    print(f"  Config A: Compute={util_a_summary.get(gpu_id, {}).get('compute_avg', 0):.1f}%, "
          f"Memory={util_a_summary.get(gpu_id, {}).get('memory_avg', 0):.4f}%")
    print(f"  Config B: Compute={util_b_summary.get(gpu_id, {}).get('compute_avg', 0):.1f}%, "
          f"Memory={util_b_summary.get(gpu_id, {}).get('memory_avg', 0):.4f}%")

### 4.5 Routing Balance

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from endpoint_metrics_helper import analyze_routing_detailed
from enhanced_metrics_visualization import visualize_routing_balance

routing_a = analyze_routing_detailed(IC_NAME_A, hours=1)
routing_b = analyze_routing_detailed(IC_NAME_B, hours=1)

visualize_routing_balance([IC_NAME_A, IC_NAME_B], [routing_a, routing_b])
    
# Print detailed summary
print("\n" + "="*100)
print("ROUTING BALANCE ANALYSIS")
print("="*100)

for ic_name, routing in [(IC_NAME_A, routing_a), (IC_NAME_B, routing_b)]:
    print(f"\n{ic_name}:")
    
    if 'model_copies' in routing:
        print(f"  Configuration: {routing['model_copies']} copies × {routing['gpus_per_copy']} GPU(s)")
    
    if 'total_invocations' in routing:
        print(f"  Total Invocations: {routing['total_invocations']['sum']:,.0f}")
    
    if 'container_invocations' in routing:
        containers = routing['container_invocations']
        totals = [c['total'] for c in containers]
        
        print(f"  Per-Copy Stats:")
        print(f"    Average: {np.mean(totals):,.0f}")
        print(f"    Min: {min(totals):,.0f}")
        print(f"    Max: {max(totals):,.0f}")
        print(f"    Std Dev: {np.std(totals):,.1f}")

print("\n" + "="*100)

### 4.6 Cost Comparison

In [ ]:
# Visualize cost comparison
import matplotlib.pyplot as plt
import numpy as np

fig, ((ax1, ax2)) = plt.subplots(2, 1, figsize=(14, 10))

# Configuration labels
config_a_label = f"{cost_copies_A['model_copies']} copies × {cost_copies_A['gpus_per_copy']:.0f} GPU"
config_b_label = f"{cost_copies_B['model_copies']} copy × {cost_copies_B['gpus_per_copy']:.0f} GPUs"
configs = [config_a_label, config_b_label]
colors = ['steelblue', 'coral']

# 1. Total Invocations Comparison
invocations = [cost_copies_A['invocations'], cost_copies_B['invocations']]
bars = ax1.bar(configs, invocations, color=colors, alpha=0.7, width=0.6)
ax1.set_ylabel('Total Invocations', fontsize=10)
ax1.set_title('Total Invocations Comparison', fontsize=11, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, invocations):
    ax1.text(bar.get_x() + bar.get_width()/2, val + max(invocations)*0.02,
            f'{val:,.0f}', ha='center', va='bottom', fontweight='bold', fontsize=9)

# 2. Cost per 1K Invocations
cost_per_1k = [cost_copies_A['cost_per_1k'], cost_copies_B['cost_per_1k']]
bars = ax2.bar(configs, cost_per_1k, color=colors, alpha=0.7, width=0.6)
ax2.set_ylabel('Cost per 1K Invocations ($)', fontsize=10)
ax2.set_title('Cost Efficiency Comparison', fontsize=11, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, cost_per_1k):
    ax2.text(bar.get_x() + bar.get_width()/2, val + max(cost_per_1k)*0.02,
            f'${val:.2f}', ha='center', va='bottom', fontweight='bold', fontsize=9)

# Add efficiency indicator
if cost_per_1k[0] < cost_per_1k[1]:
    winner_idx = 0
    savings_pct = ((cost_per_1k[1] - cost_per_1k[0]) / cost_per_1k[1]) * 100
else:
    winner_idx = 1
    savings_pct = ((cost_per_1k[0] - cost_per_1k[1]) / cost_per_1k[0]) * 100
    
ax2.text(0.5, 0.95, f'⭐ Config {"A" if winner_idx == 0 else "B"} is {savings_pct:.0f}% more cost-efficient assuming no scaling events',
        transform=ax2.transAxes, ha='center', va='top',
        bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.3),
        fontsize=9, fontweight='bold')

plt.show()

# Print cost analysis summary
print("\n" + "="*80)
print("COST ANALYSIS SUMMARY")
print("="*80)
print(f"\nConfiguration A ({config_a_label}):")
print(f"  Total invocations: {cost_copies_A['invocations']:,}")
print(f"  Cost per 1K: ${cost_copies_A['cost_per_1k']:.2f}")
print(f"  GPU utilization: {cost_copies_A['gpu_fraction']*100:.0f}%")

print(f"\nConfiguration B ({config_b_label}):")
print(f"  Total invocations: {cost_copies_B['invocations']:,}")
print(f"  Cost per 1K: ${cost_copies_B['cost_per_1k']:.2f}")
print(f"  GPU utilization: {cost_copies_B['gpu_fraction']*100:.0f}%")
print("="*80)

## Part 5: Cleanup

In [ ]:
from endpoint_metrics_helper import cleanup_endpoint

# Execute cleanup
print("\n" + "#"*60)
print("# ENDPOINT CLEANUP")
print("#"*60)

# cleanup_endpoint(
#     endpoint_name=ENDPOINT_NAME,
#     ic_names=[IC_NAME_A, IC_NAME_B],
#     endpoint_config_name=ENDPOINT_CONFIG_NAME,
#     model_names=[MODEL_A_NAME, MODEL_B_NAME]
# )

print("\n" + "="*60)
print("✅ ALL RESOURCES CLEANED UP SUCCESSFULLY")
print("="*60)